# Exploratory Data Analysis — Loan Repayment Challenge (Full)

Consolidates all EDA + feasibility analysis performed so far.

Reusable helpers live in the local `tht` package (data loading, EDA, target construction,
evaluation metrics). Import it with `import tht`.

**Join keys:**
- `loan.loanId` <-> `payment.loanId`
- `loan.clarityFraudId` <-> `clarity_underwriting_variables.underwritingid`

**Business context:** short-term, high-cost consumer installment lending (triple-digit APRs).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tht

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)


## 1. Load the data

`tht.load_data()` resolves the raw parquet path from `__file__`, so it works regardless of cwd.


In [2]:
data = tht.load_data()
df_loan = data['loan']
df_payment = data['payment']
df_underwriting = data['underwriting']

for name, df in data.items():
    print(f'{name:<13} {df.shape[0]:>9,} rows x {df.shape[1]:>2} cols')


loan            573,402 rows x 19 cols
payment         689,364 rows x  9 cols
underwriting     49,596 rows x 54 cols


## 2. Missingness overview


In [ ]:
for name, df in data.items():
    report = tht.missingness_report(df)
    print(f'\n=== {name}: {len(report)}/{df.shape[1]} columns have nulls ===')
    print(report.head(12).to_string())


=== loan: 10/19 columns have nulls ===
                n_missing  pct_missing
originatedDate     527886        92.06
fpStatus           522233        91.08
clarityFraudId     219989        38.37
apr                  3906         0.68
loanAmount           2234         0.39
payFrequency         1257         0.22
loanStatus            387         0.07
loanId                255         0.04
state                 132         0.02
nPaidOff               24         0.00

=== payment: 2/9 columns have nulls ===
                   n_missing  pct_missing
paymentReturnCode     657831        95.43
paymentStatus         164057        23.80

=== underwriting: 53/54 columns have nulls ===
                                                                                                     n_missing  pct_missing
.underwritingdataclarity.clearfraud.clearfraudidentityverification.phonetype                             48081        96.95
.underwritingdataclarity.clearfraud.clearfraudidentityverification.s

: 

## 3. Loan file — outcome & funnel

`loanStatus` is the key to building the target. Most rows never become loans.


In [ ]:
status = df_loan['loanStatus'].value_counts(dropna=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
status.head(10).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('loanStatus — top 10 by count')
axes[0].set_ylabel('count')
axes[0].tick_params(axis='x', rotation=45)

top6 = status.head(6).copy()
top6['Other'] = status.iloc[6:].sum()
top6.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
axes[1].set_title('loanStatus share (top 6 + other)')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(status.head(15).to_string())


In [ ]:
funnel = pd.DataFrame({
    'stage': ['applications', 'originated', 'approved', 'funded'],
    'count': [len(df_loan), df_loan['originated'].sum(), df_loan['approved'].sum(), df_loan['isFunded'].sum()],
})
funnel['pct_of_prev'] = (funnel['count'] / funnel['count'].shift(1) * 100).round(2)
funnel['pct_of_total'] = (funnel['count'] / len(df_loan) * 100).round(2)
funnel


## 3.1 Numeric features

Note the extreme APRs and ~2x ratio between scheduled repayment and principal.


In [ ]:
num_cols = ['apr', 'loanAmount', 'originallyScheduledPaymentAmount', 'nPaidOff', 'leadCost']
df_loan[num_cols].describe().T


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col in zip(axes, ['apr', 'loanAmount', 'originallyScheduledPaymentAmount']):
    series = df_loan[col].dropna()
    cap = series.quantile(0.99)
    series.clip(upper=cap).hist(bins=50, ax=ax, color='steelblue')
    ax.set_title(f'{col}\n(99th pct = {cap:,.0f})')
    ax.set_ylabel('count')
plt.tight_layout()
plt.show()


## 3.2 Categorical features


In [ ]:
for col in ['payFrequency', 'leadType']:
    print(f'=== {col} ===')
    print(df_loan[col].value_counts(dropna=False).head(12).to_string())
    print()

print('=== state (top 15) ===')
print(df_loan['state'].value_counts(dropna=False).head(15).to_string())


In [ ]:
print('nPaidOff — number of prior loans fully repaid (key behavioural feature)')
print(df_loan['nPaidOff'].value_counts(dropna=False).sort_index().head(25).to_string())


## 4. Payment file — repayment behaviour

This file is where the **target** and **LGD** signal come from, not application-time features.


In [ ]:
print('=== paymentStatus ===')
print(df_payment['paymentStatus'].value_counts(dropna=False).head(12).to_string())
print('\n=== isCollection (custom collection plan) ===')
print(df_payment['isCollection'].value_counts(dropna=False).to_string())
print('\n=== paymentReturnCode (top 15) ===')
print(df_payment['paymentReturnCode'].value_counts(dropna=False).head(15).to_string())


In [ ]:
n_payments = df_payment.groupby('loanId').size()
print(f'unique loans in payment file: {n_payments.shape[0]:,}')
print('payments per loan:')
print(n_payments.describe().round(1).to_string())


In [ ]:
example_loan = df_payment['loanId'].iloc[0]
ex = df_payment[df_payment['loanId'] == example_loan].sort_values('installmentIndex')
print(f'Example amortization — loan {example_loan}: {len(ex)} installments')
display(ex.head(8))

ex.plot(x='installmentIndex', y=['principal', 'fees', 'paymentAmount'], marker='o', figsize=(11, 5))
plt.title(f'Amortization schedule — {example_loan}')
plt.ylabel('amount ($)')
plt.xlabel('installmentIndex')
plt.show()


## 5. Underwriting file — fraud / identity (third-party Clarity data)

**Coverage nuance:** clarity data covers only ~9% of *all* applications, but **~83% of resolved (funded) loans**.


In [ ]:
n_with_cf = df_loan['clarityFraudId'].notna().sum()
print(f'loans with clarityFraudId:  {n_with_cf:,} ({n_with_cf/len(df_loan)*100:.1f}%) of ALL applications')
print(f'underwriting report rows:    {len(df_underwriting):,}')
print(f'unique underwriting ids:     {df_underwriting["underwritingid"].nunique():,}')
print()
print('clearFraudScore (higher = lower default risk):')
print(df_underwriting['clearfraudscore'].describe().to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_underwriting['clearfraudscore'].hist(bins=50, ax=axes[0], color='tomato')
axes[0].set_title('clearFraudScore distribution')
axes[0].set_xlabel('score (higher = lower default risk)')

total_col = [c for c in df_underwriting.columns if 'totalnumberoffraudindicators' in c]
if total_col:
    df_underwriting[total_col[0]].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='tomato')
    axes[1].set_title('Total number of fraud indicators')
    axes[1].set_xlabel('count')
plt.tight_layout()
plt.show()


In [ ]:
sparsity = tht.fraud_indicator_sparsity(df_underwriting)
print(f'{len(sparsity)} fraud-indicator columns. Positive (True) rate:')
print(sparsity.to_string())
print()
print(f'mean positive rate across indicators: {sparsity.mean():.4f}')
print('(near-constant indicators, e.g. ~0 or ~1, carry little signal and should be dropped)')


## 6. Target construction

`tht.build_target()` applies the preliminary default label. Refined later with payment return codes.


In [ ]:
outcome = tht.build_target(df_loan)

print(f'Loans with definitive repayment outcome: {len(outcome):,} of {len(df_loan):,} total')
print(f"  Good (Paid Off):      {(outcome['defaulted']==0).sum():,}")
print(f"  Bad  (defaulted):     {(outcome['defaulted']==1).sum():,}")
print(f"  Default rate:         {outcome['defaulted'].mean()*100:.1f}%")


## 6.1 Default rate by segment — the strongest application-time signals


In [ ]:
tht.segment_default_rate(outcome, 'leadType')


In [ ]:
outcome = outcome.assign(
    npo=pd.cut(outcome['nPaidOff'], [-1, 0, 1, 5, 1000], labels=['0', '1', '2-5', '6+'])
)
tht.segment_default_rate(outcome, 'npo')


## 6.2 Predictive power of the fraud score (single feature)

A quick AUC of `clearfraudscore` against the default label, on the resolved-and-covered subset.


In [ ]:
u = outcome[outcome['hasCF'] == 1].merge(
    df_underwriting[['underwritingid', 'clearfraudscore']],
    left_on='clarityFraudId', right_on='underwritingid', how='inner',
).dropna(subset=['clearfraudscore', 'defaulted'])

print(f'resolved & hasCF rows: {len(u):,} | default rate: {u["defaulted"].mean():.3f}')
print(f'coverage among resolved: {(outcome["hasCF"]==1).mean()*100:.1f}%')
print(f'AUC(clearfraudscore -> default): {tht.auc(u["defaulted"], u["clearfraudscore"]):.3f}')


## 7. Key findings

1. **Most applications never become loans** — ~78% `Withdrawn Application`, ~15% `Rejected`; only ~7% originate.
2. **High-cost, short-term lending** — APRs of 360-590% and scheduled repayment ~ 2x principal.
3. **Resolved cohort is small but adequate** — 28,524 loans with a definitive outcome, 60.2% default rate.
4. **`nPaidOff` is monotonic and strong** — 0 -> 66% default, 1 -> 41%, 2-5 -> 30%, 6+ -> 9.5%.
5. **`leadType` separates strongly** — `bvMandatory` 68% default vs `express` 10%.
6. **Fraud data is well-covered where it matters** — only 9% of all applications, but 83% of resolved loans.
7. **`clearfraudscore` is a strong single feature** — AUC 0.634 against default on its own.
8. **Fraud indicators are sparse** (mean 6.9% positive); one near-constant (92% true) should be dropped.

### Feasibility vs a 70% (AUC) bar
- **PD:** supported — full feature set should clear AUC >= 0.70.
- **Fraud model:** not measurable — no ground-truth fraud label.
- **LGD:** under-powered — ~17k defaults, noisy target; 70% R^2 unrealistic.
- **EAD / eligibility:** N/A — fixed-term loans / deterministic rules, not ML.
